<a href="https://colab.research.google.com/github/marcocintra/Atmosphere/blob/master/GOPI_TEC_SJC_ANALYSIS_DECEMBER_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GOPI - DECEMBER/2024

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import glob

data_dir = '.'

std_files = glob.glob(os.path.join(data_dir, 'sjsp3*-2024-12-*.Std'))

if not std_files:
    std_files = glob.glob(os.path.join(data_dir, 'sjsp3*-2024-12-*'))

std_files.sort()

dfs = []

for file_path in std_files:
    try:

        filename = os.path.basename(file_path)
        print(f"Processando {filename}")

        date_parts = filename.replace('.Std', '').split('-')
        year = int(date_parts[1])
        month = int(date_parts[2])
        day = int(date_parts[3])
        base_date = datetime(year, month, day)

        df = pd.read_csv(file_path, sep='\s+', header=None)

        df.columns = ['time_ut', 'tec', 'tec_std', 'latitude']

        def decimal_to_time(decimal_hours):
            hours = int(decimal_hours)
            minutes = int((decimal_hours - hours) * 60)
            seconds = int(((decimal_hours - hours) * 60 - minutes) * 60)
            return base_date + timedelta(hours=hours, minutes=minutes, seconds=seconds)

        df['DATETIME'] = df['time_ut'].apply(decimal_to_time)

        df['tec'] = pd.to_numeric(df['tec'].replace('-', np.nan))
        df['tec_std'] = pd.to_numeric(df['tec_std'].replace('-', np.nan))

        df = df[['DATETIME', 'tec', 'tec_std', 'latitude']]
        df.columns = ['DATETIME', 'TEC', 'TEC_STD', 'LATITUDE']

        dfs.append(df)

    except Exception as e:
        print(f"Erro ao processar o arquivo {file_path}: {e}")

if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)

    combined_df = combined_df.sort_values('DATETIME')

    combined_df.to_pickle('sjsp_dezembro_2024_completo.pkl')

    print(f"DataFrame combinado criado com {len(combined_df)} linhas")
    print("Primeiras 5 linhas:")
    print(combined_df.head())

    print("\nPeríodo coberto pelos dados:")
    print(f"Início: {combined_df['DATETIME'].min()}")
    print(f"Fim: {combined_df['DATETIME'].max()}")
    print(f"Total de dias: {(combined_df['DATETIME'].max() - combined_df['DATETIME'].min()).days + 1}")

    dias_presentes = combined_df['DATETIME'].dt.date.unique()
    print(f"\nTotal de dias com dados: {len(dias_presentes)}")
    print("Dias presentes:", sorted(dias_presentes))
else:
    print("Nenhum arquivo foi processado com sucesso.")

Processando sjsp336-2024-12-01.Std
Processando sjsp337-2024-12-02.Std
Processando sjsp338-2024-12-03.Std
Processando sjsp339-2024-12-04.Std
Processando sjsp340-2024-12-05.Std
Processando sjsp341-2024-12-06.Std
Processando sjsp342-2024-12-07.Std
Processando sjsp348-2024-12-13.Std
Processando sjsp349-2024-12-14.Std
Processando sjsp350-2024-12-15.Std
Processando sjsp351-2024-12-16.Std
Processando sjsp352-2024-12-17.Std
Processando sjsp353-2024-12-18.Std
Processando sjsp354-2024-12-19.Std
Processando sjsp355-2024-12-20.Std
Processando sjsp356-2024-12-21.Std
Processando sjsp357-2024-12-22.Std
Processando sjsp358-2024-12-23.Std
Processando sjsp359-2024-12-24.Std
Processando sjsp360-2024-12-25.Std
Processando sjsp361-2024-12-26.Std
Processando sjsp362-2024-12-27.Std
Processando sjsp363-2024-12-28.Std
Processando sjsp364-2024-12-29.Std
Processando sjsp365-2024-12-30.Std
Processando sjsp366-2024-12-31.Std
DataFrame combinado criado com 36180 linhas
Primeiras 5 linhas:
             DATETIME    T

In [ ]:
combined_df

,DATETIME,TEC,TEC_STD,LATITUDE
0,2024-12-01 00:00:00,50.09,23.77,-23.21
1,2024-12-01 00:01:01,50.02,22.40,-23.21
2,2024-12-01 00:01:58,49.94,22.07,-23.21
3,2024-12-01 00:03:00,49.86,19.59,-23.21
4,2024-12-01 00:04:01,49.77,21.47,-23.21
...,...,...,...,...
36175,2024-12-31 23:55:01,31.16,6.37,-23.21
36176,2024-12-31 23:55:58,31.14,6.43,-23.21
36177,2024-12-31 23:56:59,31.13,6.44,-23.21
36178,2024-12-31 23:58:01,31.11,6.50,-23.21


## 18:00

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta, time

df = pd.read_pickle('sjsp_dezembro_2024_completo.pkl')
print(f"Dados carregados: {len(df)} registros de {df['DATETIME'].min().date()} até {df['DATETIME'].max().date()}")

df = df.sort_values('DATETIME').reset_index(drop=True)

df_clean = df.dropna(subset=['TEC', 'TEC_STD']).copy()
print(f"Analisando {len(df_clean)} pontos após remover valores ausentes")

def check_consecutive_highs(df, peak_datetime, threshold_percent=90):

    peak_idx = df[df['DATETIME'] == peak_datetime].index[0]

    peak_value = df.loc[peak_idx, 'TEC']

    threshold = peak_value * (threshold_percent / 100)

    idx_before = peak_idx - 1
    idx_after = peak_idx + 1

    valid_before = idx_before >= 0
    valid_after = idx_after < len(df)

    if valid_before and valid_after:
        values = [df.loc[idx_before, 'TEC'], peak_value, df.loc[idx_after, 'TEC']]
        stds = [df.loc[idx_before, 'TEC_STD'], df.loc[peak_idx, 'TEC_STD'], df.loc[idx_after, 'TEC_STD']]
        datetimes = [df.loc[idx_before, 'DATETIME'], peak_datetime, df.loc[idx_after, 'DATETIME']]
        all_high = all(value >= threshold for value in values)

        if all_high:
            return {'indices': [idx_before, peak_idx, idx_after],
                   'values': values,
                   'stds': stds,
                   'datetimes': datetimes}

    if valid_before and idx_before > 0:
        values = [df.loc[idx_before-1, 'TEC'], df.loc[idx_before, 'TEC'], peak_value]
        stds = [df.loc[idx_before-1, 'TEC_STD'], df.loc[idx_before, 'TEC_STD'], df.loc[peak_idx, 'TEC_STD']]
        datetimes = [df.loc[idx_before-1, 'DATETIME'], df.loc[idx_before, 'DATETIME'], peak_datetime]
        all_high = all(value >= threshold for value in values)

        if all_high:
            return {'indices': [idx_before-1, idx_before, peak_idx],
                   'values': values,
                   'stds': stds,
                   'datetimes': datetimes}

    if valid_after and idx_after < len(df) - 1:
        values = [peak_value, df.loc[idx_after, 'TEC'], df.loc[idx_after+1, 'TEC']]
        stds = [df.loc[peak_idx, 'TEC_STD'], df.loc[idx_after, 'TEC_STD'], df.loc[idx_after+1, 'TEC_STD']]
        datetimes = [peak_datetime, df.loc[idx_after, 'DATETIME'], df.loc[idx_after+1, 'DATETIME']]
        all_high = all(value >= threshold for value in values)

        if all_high:
            return {'indices': [peak_idx, idx_after, idx_after+1],
                   'values': values,
                   'stds': stds,
                   'datetimes': datetimes}

    return None

def analyze_peaks_at_precise_interval(hour_start, minute_start, second_start, hour_end, minute_end, second_end):

    interval_label = f"{hour_start:02d}:{minute_start:02d}:{second_start:02d}-{hour_end:02d}:{minute_end:02d}:{second_end:02d}"

    print(f"\n{'='*80}")
    print(f"ANÁLISE DE PICOS DE TEC NO INTERVALO PRECISO {interval_label} UTC")
    print(f"{'='*80}")

    start_timestamps = []
    end_timestamps = []

    for day in df_clean['DATETIME'].dt.date.unique():
        start_timestamps.append(pd.Timestamp(day.year, day.month, day.day, hour_start, minute_start, second_start))
        end_timestamps.append(pd.Timestamp(day.year, day.month, day.day, hour_end, minute_end, second_end))

    df_interval = pd.DataFrame()
    for start, end in zip(start_timestamps, end_timestamps):
        day_interval = df_clean[(df_clean['DATETIME'] >= start) & (df_clean['DATETIME'] <= end)]
        df_interval = pd.concat([df_interval, day_interval])

    print(f"Filtrando para horários entre {interval_label} UTC: {len(df_interval)} pontos restantes")

    if len(df_interval) == 0:
        print(f"ATENÇÃO: Não foram encontrados dados no intervalo {interval_label} UTC!")
        return pd.DataFrame(), []

    df_interval['DATE'] = df_interval['DATETIME'].dt.date

    daily_max = df_interval.loc[df_interval.groupby('DATE')['TEC'].idxmax()]
    print(f"Dias únicos com dados no intervalo {interval_label}: {len(daily_max)}")

    if len(daily_max) >= 5:
        top_5_days = daily_max.nlargest(5, 'TEC').reset_index(drop=True)
    else:
        top_5_days = daily_max.sort_values('TEC', ascending=False).reset_index(drop=True)
        print(f"ATENÇÃO: Apenas {len(top_5_days)} dias possuem dados neste intervalo de tempo!")

    print(f"\n{len(top_5_days)} MAIORES VALORES DE TEC NO INTERVALO {interval_label} (UM POR DIA):")
    for idx, row in top_5_days.iterrows():
        print(f"{idx+1}. TEC: {row['TEC']:.2f} ± {row['TEC_STD']:.2f} TECU, "
              f"Data/Hora: {row['DATETIME']}, Latitude: {row['LATITUDE']}")

    print(f"\nANÁLISE DE SEQUÊNCIAS (Valores permanecem acima de 90% do pico em 3 pontos consecutivos):")
    sequences = []

    for idx, row in top_5_days.iterrows():

        sequence = check_consecutive_highs(df_clean, row['DATETIME'], threshold_percent=90)

        if sequence:
            sequences.append({
                'peak_rank': idx + 1,
                'peak_value': row['TEC'],
                'peak_std': row['TEC_STD'],
                'peak_datetime': row['DATETIME'],
                'sequence': sequence
            })

            print(f"\nPico #{idx+1} (TEC = {row['TEC']:.2f} ± {row['TEC_STD']:.2f}) tem sequência de 3 valores altos:")
            for i, (dt, val, std) in enumerate(zip(sequence['datetimes'], sequence['values'], sequence['stds'])):
                print(f"  {i+1}. {dt}: {val:.2f} ± {std:.2f} TECU ({val/row['TEC']*100:.1f}% do pico)")
        else:
            print(f"\nPico #{idx+1} (TEC = {row['TEC']:.2f} ± {row['TEC_STD']:.2f}) "
                  f"NÃO tem sequência de 3 valores altos acima de 90% do pico")

    if len(top_5_days) > 0:
        print(f"\nTABELA RESUMO DOS PICOS DE TEC NO INTERVALO {interval_label} UTC:")
        print("-" * 100)
        print(f"{'Rank':<5}{'Data':<12}{'Hora UTC':<12}{'TEC':<10}{'TEC_STD':<10}{'Sequência':<15}{'Duração da Sequência'}")
        print("-" * 100)

        for idx, row in top_5_days.iterrows():
            has_sequence = any(seq['peak_datetime'] == row['DATETIME'] for seq in sequences)
            sequence_info = "Sim" if has_sequence else "Não"

            duration = ""
            if has_sequence:
                for seq in sequences:
                    if seq['peak_datetime'] == row['DATETIME']:
                        first_dt = min(seq['sequence']['datetimes'])
                        last_dt = max(seq['sequence']['datetimes'])
                        duration = f"{(last_dt - first_dt).total_seconds() / 60:.1f} min"
                        break

            print(f"{idx+1:<5}{row['DATETIME'].strftime('%d/%m/%Y'):<12}{row['DATETIME'].strftime('%H:%M:%S'):<12}"
                  f"{row['TEC']:.2f}    {row['TEC_STD']:.2f}    {sequence_info:<15}{duration}")
        print("-" * 100)

    return top_5_days, sequences

def analyze_peaks_at_interval(hour_start, minute_start, hour_end, minute_end):

    return analyze_peaks_at_precise_interval(hour_start, minute_start, 0, hour_end, minute_end, 0)

print("\n\n" + "*"*40 + " ANÁLISE DO INTERVALO EXATO 17:59:00-18:01:00 UTC " + "*"*40)
top_5_18h_exato, sequences_18h_exato = analyze_peaks_at_precise_interval(17, 59, 55, 18, 0, 5)

print("\nAnálise concluída!")

Dados carregados: 36180 registros de 2024-12-01 até 2024-12-31
Analisando 36172 pontos após remover valores ausentes


**************************************** ANÁLISE DO INTERVALO EXATO 17:59:00-18:01:00 UTC ****************************************

ANÁLISE DE PICOS DE TEC NO INTERVALO PRECISO 17:59:55-18:00:05 UTC
Filtrando para horários entre 17:59:55-18:00:05 UTC: 24 pontos restantes
Dias únicos com dados no intervalo 17:59:55-18:00:05: 24

5 MAIORES VALORES DE TEC NO INTERVALO 17:59:55-18:00:05 (UM POR DIA):
1. TEC: 71.73 ± 5.98 TECU, Data/Hora: 2024-12-18 18:00:00, Latitude: -23.21
2. TEC: 67.49 ± 4.86 TECU, Data/Hora: 2024-12-22 18:00:00, Latitude: -23.21
3. TEC: 67.35 ± 2.73 TECU, Data/Hora: 2024-12-19 18:00:00, Latitude: -23.21
4. TEC: 67.11 ± 2.09 TECU, Data/Hora: 2024-12-01 18:00:00, Latitude: -23.21
5. TEC: 64.56 ± 2.65 TECU, Data/Hora: 2024-12-20 18:00:00, Latitude: -23.21

ANÁLISE DE SEQUÊNCIAS (Valores permanecem acima de 90% do pico em 3 pontos consecutivos):

Pico #1 (

## DECEMBER

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta, time

df = pd.read_pickle('sjsp_dezembro_2024_completo.pkl')
print(f"Dados carregados: {len(df)} registros de {df['DATETIME'].min().date()} até {df['DATETIME'].max().date()}")

df = df.sort_values('DATETIME').reset_index(drop=True)

df_clean = df.dropna(subset=['TEC', 'TEC_STD']).copy()
print(f"Analisando {len(df_clean)} pontos após remover valores ausentes")

def check_consecutive_highs(df, peak_datetime, threshold_percent=90):

    peak_idx = df[df['DATETIME'] == peak_datetime].index[0]

    peak_value = df.loc[peak_idx, 'TEC']

    threshold = peak_value * (threshold_percent / 100)

    idx_before = peak_idx - 1
    idx_after = peak_idx + 1

    valid_before = idx_before >= 0
    valid_after = idx_after < len(df)

    if valid_before and valid_after:
        values = [df.loc[idx_before, 'TEC'], peak_value, df.loc[idx_after, 'TEC']]
        stds = [df.loc[idx_before, 'TEC_STD'], df.loc[peak_idx, 'TEC_STD'], df.loc[idx_after, 'TEC_STD']]
        datetimes = [df.loc[idx_before, 'DATETIME'], peak_datetime, df.loc[idx_after, 'DATETIME']]
        all_high = all(value >= threshold for value in values)

        if all_high:
            return {'indices': [idx_before, peak_idx, idx_after],
                   'values': values,
                   'stds': stds,
                   'datetimes': datetimes}

    if valid_before and idx_before > 0:
        values = [df.loc[idx_before-1, 'TEC'], df.loc[idx_before, 'TEC'], peak_value]
        stds = [df.loc[idx_before-1, 'TEC_STD'], df.loc[idx_before, 'TEC_STD'], df.loc[peak_idx, 'TEC_STD']]
        datetimes = [df.loc[idx_before-1, 'DATETIME'], df.loc[idx_before, 'DATETIME'], peak_datetime]
        all_high = all(value >= threshold for value in values)

        if all_high:
            return {'indices': [idx_before-1, idx_before, peak_idx],
                   'values': values,
                   'stds': stds,
                   'datetimes': datetimes}

    if valid_after and idx_after < len(df) - 1:
        values = [peak_value, df.loc[idx_after, 'TEC'], df.loc[idx_after+1, 'TEC']]
        stds = [df.loc[peak_idx, 'TEC_STD'], df.loc[idx_after, 'TEC_STD'], df.loc[idx_after+1, 'TEC_STD']]
        datetimes = [peak_datetime, df.loc[idx_after, 'DATETIME'], df.loc[idx_after+1, 'DATETIME']]
        all_high = all(value >= threshold for value in values)

        if all_high:
            return {'indices': [peak_idx, idx_after, idx_after+1],
                   'values': values,
                   'stds': stds,
                   'datetimes': datetimes}

    return None

def analyze_top_peaks(top_n=200):

    print(f"\n{'='*80}")
    print(f"ANÁLISE DOS TOP {top_n} PICOS DE TEC (TODOS OS HORÁRIOS)")
    print(f"{'='*80}")

    df_clean['DATE'] = df_clean['DATETIME'].dt.date

    top_n_peaks = df_clean.nlargest(top_n, 'TEC').reset_index(drop=True)

    print(f"\nTOP {top_n} MAIORES VALORES DE TEC (TODOS OS HORÁRIOS):")
    for idx, row in top_n_peaks.iterrows():
        print(f"{idx+1}. TEC: {row['TEC']:.2f} ± {row['TEC_STD']:.2f} TECU, "
              f"Data/Hora: {row['DATETIME']}, Latitude: {row['LATITUDE']}")

    print(f"\nANÁLISE DE SEQUÊNCIAS (Valores permanecem acima de 90% do pico em 3 pontos consecutivos):")
    sequences = []

    for idx, row in top_n_peaks.iterrows():

        sequence = check_consecutive_highs(df_clean, row['DATETIME'], threshold_percent=90)

        if sequence:
            sequences.append({
                'peak_rank': idx + 1,
                'peak_value': row['TEC'],
                'peak_std': row['TEC_STD'],
                'peak_datetime': row['DATETIME'],
                'sequence': sequence
            })

            print(f"\nPico #{idx+1} (TEC = {row['TEC']:.2f} ± {row['TEC_STD']:.2f}) tem sequência de 3 valores altos:")
            for i, (dt, val, std) in enumerate(zip(sequence['datetimes'], sequence['values'], sequence['stds'])):
                print(f"  {i+1}. {dt}: {val:.2f} ± {std:.2f} TECU ({val/row['TEC']*100:.1f}% do pico)")
        else:
            print(f"\nPico #{idx+1} (TEC = {row['TEC']:.2f} ± {row['TEC_STD']:.2f}) "
                  f"NÃO tem sequência de 3 valores altos acima de 90% do pico")

    if len(top_n_peaks) > 0:
        print(f"\nTABELA RESUMO DOS TOP {top_n} PICOS DE TEC:")
        print("-" * 110)
        print(f"{'Rank':<5}{'Data':<12}{'Hora UTC':<10}{'TEC':<10}{'TEC_STD':<10}{'Sequência':<15}{'Duração da Sequência':<20}{'Horário do dia'}")
        print("-" * 110)

        for idx, row in top_n_peaks.iterrows():
            has_sequence = any(seq['peak_datetime'] == row['DATETIME'] for seq in sequences)
            sequence_info = "Sim" if has_sequence else "Não"

            duration = ""
            if has_sequence:
                for seq in sequences:
                    if seq['peak_datetime'] == row['DATETIME']:
                        first_dt = min(seq['sequence']['datetimes'])
                        last_dt = max(seq['sequence']['datetimes'])
                        duration = f"{(last_dt - first_dt).total_seconds() / 60:.1f} min"
                        break

            hour = row['DATETIME'].hour
            minute = row['DATETIME'].minute
            time_category = f"{hour:02d}:{minute:02d} UTC"

            print(f"{idx+1:<5}{row['DATETIME'].strftime('%d/%m/%Y'):<12}{row['DATETIME'].strftime('%H:%M'):<10}"
                  f"{row['TEC']:.2f}    {row['TEC_STD']:.2f}    {sequence_info:<15}{duration:<20}{time_category}")
        print("-" * 110)

    return top_n_peaks, sequences

print("\n\n" + "*"*40 + " ANÁLISE DOS TOP 200 PICOS DE TEC (TODOS OS HORÁRIOS) " + "*"*40)
top_n_peaks, sequences = analyze_top_peaks(top_n=200)

print("\nAnálise concluída!")

Dados carregados: 36180 registros de 2024-12-01 até 2024-12-31
Analisando 36172 pontos após remover valores ausentes


**************************************** ANÁLISE DOS TOP 200 PICOS DE TEC (TODOS OS HORÁRIOS) ****************************************

ANÁLISE DOS TOP 200 PICOS DE TEC (TODOS OS HORÁRIOS)

TOP 200 MAIORES VALORES DE TEC (TODOS OS HORÁRIOS):
1. TEC: 73.56 ± 8.07 TECU, Data/Hora: 2024-12-18 18:16:58, Latitude: -23.21
2. TEC: 73.55 ± 7.94 TECU, Data/Hora: 2024-12-18 18:16:01, Latitude: -23.21
3. TEC: 73.55 ± 8.22 TECU, Data/Hora: 2024-12-18 18:18:00, Latitude: -23.21
4. TEC: 73.53 ± 7.79 TECU, Data/Hora: 2024-12-18 18:15:00, Latitude: -23.21
5. TEC: 73.53 ± 8.35 TECU, Data/Hora: 2024-12-18 18:19:01, Latitude: -23.21
6. TEC: 73.49 ± 7.65 TECU, Data/Hora: 2024-12-18 18:13:58, Latitude: -23.21
7. TEC: 73.49 ± 8.34 TECU, Data/Hora: 2024-12-18 18:19:58, Latitude: -23.21
8. TEC: 73.44 ± 8.71 TECU, Data/Hora: 2024-12-18 18:21:00, Latitude: -23.21
9. TEC: 73.43 ± 7.51 TECU, Data

# IGS

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
%cd /content

/content


In [ ]:
#df_mapas_igs_dec_2024.pkl
!gdown '1T0gBaH6IyIX72T1VPJuZa2D43HCrRFVp'

Downloading...
From: https://drive.google.com/uc?id=1T0gBaH6IyIX72T1VPJuZa2D43HCrRFVp
To: /content/df_mapas_igs_dec_2024.pkl
100% 3.60M/3.60M [00:00<00:00, 221MB/s]


In [ ]:
df_mapas_igs_2024 = pd.read_pickle('/content/df_mapas_igs_dec_2024.pkl')

In [ ]:
df_mapas_igs_2024

,DATETIME,TECMAP
0,2024-12-01 00:00:00,"[[37.7, 34.6, 31.9, 30.0, 28.6, 27.2, 25.8, 24..."
1,2024-12-01 02:00:00,"[[20.3, 18.7, 18.1, 18.2, 18.2, 18.1, 17.7, 17..."
2,2024-12-01 04:00:00,"[[11.6, 11.8, 12.0, 12.4, 13.0, 13.5, 14.0, 14..."
3,2024-12-01 06:00:00,"[[10.4, 10.8, 11.1, 11.7, 12.1, 12.7, 13.3, 13..."
4,2024-12-01 08:00:00,"[[9.9, 10.1, 10.3, 10.6, 11.0, 11.6, 12.3, 13...."
...,...,...
343,2024-12-30 14:00:00,"[[8.9, 11.1, 14.4, 18.8, 23.7, 28.3, 32.7, 36...."
344,2024-12-30 16:00:00,"[[26.3, 29.8, 33.4, 37.1, 40.5, 43.7, 47.0, 49..."
345,2024-12-30 18:00:00,"[[43.5, 44.5, 45.3, 46.2, 47.1, 48.6, 50.0, 51..."
346,2024-12-30 20:00:00,"[[48.8, 49.1, 49.0, 48.7, 48.5, 48.1, 48.0, 47..."


## 12/18/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-18'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_igs_2024[df_mapas_igs_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:

    df_temp = df_mapas_igs_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_igs_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_igs_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
201,2024-12-18 18:00:00,"[[38.6, 42.2, 45.2, 47.7, 49.5, 50.3, 49.9, 49..."


In [ ]:
mapas_igs = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_igs = []
for i in range(len(mapas_igs)):
    np_mapas_igs.append(mapas_igs[i])
np_mapas_igs = np.array(np_mapas_igs)

In [ ]:
np_mapas_igs

array([[[38.6, 42.2, 45.2, ..., 15.9, 15.3, 15.4],
        [38.9, 42.4, 45.9, ..., 18.4, 17.5, 17.4],
        [39.1, 42.5, 46.1, ..., 23.7, 22. , 21. ],
        ...,
        [24.3, 24. , 23.8, ..., 19.1, 18.4, 17.9],
        [24.1, 23.9, 23.6, ..., 18.6, 18.1, 17.7],
        [23.5, 23.3, 23.1, ..., 18.1, 17.6, 17.3]]])

In [ ]:
np_mapas_igs.shape

(1, 49, 23)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class Igs(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        super().__init__(extent, lat_step, lon_step)


igs_tec = Igs(extent=(-110, 0, -80, 40), lat_step=2.5, lon_step=5)
igs_points = igs_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(igs_points)

[(-25.0, -50, 22, 12), (-25.0, -45, 22, 13), (-22.5, -50, 23, 12), (-22.5, -45, 23, 13)]


In [ ]:
igs_points

[(-25.0, -50, 22, 12),
 (-25.0, -45, 22, 13),
 (-22.5, -50, 23, 12),
 (-22.5, -45, 23, 13)]

In [ ]:
igs_tec.tec_map = np_mapas_igs[0]

print("Os 4 pontos mais próximos são:")
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = igs_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(igs_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in igs_points]
lons = [p[1] for p in igs_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(igs_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-25.0°, -50°) - Índices: [22, 12] - Valor TEC: 93.5
Coordenadas: (-25.0°, -45°) - Índices: [22, 13] - Valor TEC: 94.5
Coordenadas: (-22.5°, -50°) - Índices: [23, 12] - Valor TEC: 93.4
Coordenadas: (-22.5°, -45°) - Índices: [23, 13] - Valor TEC: 93.6

Valor TEC interpolado para (-23.21°, -45.51°): 93.8120


## 12/22/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-22'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_igs_2024[df_mapas_igs_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_igs_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_igs_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_igs_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
249,2024-12-22 18:00:00,"[[48.9, 51.6, 53.7, 54.6, 54.9, 54.7, 54.6, 54..."


In [ ]:
mapas_igs = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_igs = []
for i in range(len(mapas_igs)):
    np_mapas_igs.append(mapas_igs[i])
np_mapas_igs = np.array(np_mapas_igs)

In [ ]:
np_mapas_igs

array([[[48.9, 51.6, 53.7, ..., 23. , 21.7, 20.7],
        [49.6, 52. , 53.9, ..., 25.5, 24.4, 23. ],
        [50.2, 52.1, 53.8, ..., 30.7, 29.2, 27.9],
        ...,
        [24.7, 24.7, 24.4, ..., 20.9, 20.5, 20.2],
        [25. , 24.9, 24.8, ..., 21.5, 21. , 20.8],
        [24.7, 24.8, 24.5, ..., 21.6, 21.2, 20.9]]])

In [ ]:
np_mapas_igs.shape

(1, 49, 23)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class Igs(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        super().__init__(extent, lat_step, lon_step)


igs_tec = Igs(extent=(-110, 0, -80, 40), lat_step=2.5, lon_step=5)
igs_points = igs_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(igs_points)

[(-25.0, -50, 22, 12), (-25.0, -45, 22, 13), (-22.5, -50, 23, 12), (-22.5, -45, 23, 13)]


In [ ]:
igs_points

[(-25.0, -50, 22, 12),
 (-25.0, -45, 22, 13),
 (-22.5, -50, 23, 12),
 (-22.5, -45, 23, 13)]

In [ ]:
igs_tec.tec_map = np_mapas_igs[0]

print("Os 4 pontos mais próximos são:")
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = igs_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(igs_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in igs_points]
lons = [p[1] for p in igs_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(igs_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-25.0°, -50°) - Índices: [22, 12] - Valor TEC: 92.5
Coordenadas: (-25.0°, -45°) - Índices: [22, 13] - Valor TEC: 92.7
Coordenadas: (-22.5°, -50°) - Índices: [23, 12] - Valor TEC: 91.9
Coordenadas: (-22.5°, -45°) - Índices: [23, 13] - Valor TEC: 91.5

Valor TEC interpolado para (-23.21°, -45.51°): 91.8642


## 12/19/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-19'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_igs_2024[df_mapas_igs_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_igs_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_igs_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_igs_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
213,2024-12-19 18:00:00,"[[36.1, 39.3, 41.5, 42.6, 42.4, 41.2, 39.2, 37..."


In [ ]:
mapas_igs = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_igs = []
for i in range(len(mapas_igs)):
    np_mapas_igs.append(mapas_igs[i])
np_mapas_igs = np.array(np_mapas_igs)

In [ ]:
np_mapas_igs

array([[[36.1, 39.3, 41.5, ..., 15. , 14.4, 14.1],
        [37.3, 40. , 42.4, ..., 17. , 16.5, 16.2],
        [38.2, 40.5, 42.6, ..., 20.6, 20.1, 19.7],
        ...,
        [23.4, 23.3, 23.2, ..., 23.6, 23.2, 22.9],
        [23.3, 23.2, 23.1, ..., 23.7, 23.5, 23.2],
        [23. , 23.1, 22.9, ..., 23.5, 23.6, 23.5]]])

In [ ]:
np_mapas_igs.shape

(1, 49, 23)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):
        """
        Função para encontrar os 4 pontos da grade mais próximos
        a uma coordenada específica
        """
        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class Igs(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        super().__init__(extent, lat_step, lon_step)


igs_tec = Igs(extent=(-110, 0, -80, 40), lat_step=2.5, lon_step=5)
igs_points = igs_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(igs_points)

[(-25.0, -50, 22, 12), (-25.0, -45, 22, 13), (-22.5, -50, 23, 12), (-22.5, -45, 23, 13)]


In [ ]:
igs_points

[(-25.0, -50, 22, 12),
 (-25.0, -45, 22, 13),
 (-22.5, -50, 23, 12),
 (-22.5, -45, 23, 13)]

In [ ]:
igs_tec.tec_map = np_mapas_igs[0]

print("Os 4 pontos mais próximos são:")
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = igs_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(igs_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in igs_points]
lons = [p[1] for p in igs_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(igs_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-25.0°, -50°) - Índices: [22, 12] - Valor TEC: 83.2
Coordenadas: (-25.0°, -45°) - Índices: [22, 13] - Valor TEC: 83.7
Coordenadas: (-22.5°, -50°) - Índices: [23, 12] - Valor TEC: 83.8
Coordenadas: (-22.5°, -45°) - Índices: [23, 13] - Valor TEC: 83.6

Valor TEC interpolado para (-23.21°, -45.51°): 83.6285


## 12/01/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-01'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_igs_2024[df_mapas_igs_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_igs_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_igs_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_igs_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
9,2024-12-01 18:00:00,"[[50.5, 53.3, 54.8, 55.0, 54.6, 54.1, 54.0, 54..."


In [ ]:
mapas_igs = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_igs = []
for i in range(len(mapas_igs)):
    np_mapas_igs.append(mapas_igs[i])
np_mapas_igs = np.array(np_mapas_igs)

In [ ]:
np_mapas_igs

array([[[50.5, 53.3, 54.8, ..., 26.7, 24.5, 22.9],
        [50.2, 52.6, 54.1, ..., 30.3, 28.1, 26.2],
        [50.4, 52.2, 53.4, ..., 36.6, 34. , 31.8],
        ...,
        [30.4, 30.4, 30.2, ..., 23.1, 22.6, 22. ],
        [30.3, 30.3, 30.1, ..., 23.2, 22.9, 22.5],
        [29.4, 29.3, 29.2, ..., 23.3, 23.1, 22.8]]])

In [ ]:
np_mapas_igs.shape

(1, 49, 23)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):
        """
        Função para encontrar os 4 pontos da grade mais próximos
        a uma coordenada específica
        """
        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class Igs(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        super().__init__(extent, lat_step, lon_step)


igs_tec = Igs(extent=(-110, 0, -80, 40), lat_step=2.5, lon_step=5)
igs_points = igs_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(igs_points)

[(-25.0, -50, 22, 12), (-25.0, -45, 22, 13), (-22.5, -50, 23, 12), (-22.5, -45, 23, 13)]


In [ ]:
igs_points

[(-25.0, -50, 22, 12),
 (-25.0, -45, 22, 13),
 (-22.5, -50, 23, 12),
 (-22.5, -45, 23, 13)]

In [ ]:
igs_tec.tec_map = np_mapas_igs[0]

print("Os 4 pontos mais próximos são:")
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = igs_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(igs_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in igs_points]
lons = [p[1] for p in igs_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(igs_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-25.0°, -50°) - Índices: [22, 12] - Valor TEC: 95.6
Coordenadas: (-25.0°, -45°) - Índices: [22, 13] - Valor TEC: 94.9
Coordenadas: (-22.5°, -50°) - Índices: [23, 12] - Valor TEC: 93.9
Coordenadas: (-22.5°, -45°) - Índices: [23, 13] - Valor TEC: 92.7

Valor TEC interpolado para (-23.21°, -45.51°): 93.4327


# NAGOYA

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#df_mapas_nagoya_dec_2024.pkl
!gdown '1CXL-btSyeELqYjLGL-B2oSnMt-QlPFUR&confirm=True'

In [ ]:
df_mapas_nagoya_2024 = pd.read_pickle('/content/df_mapas_nagoya_dec_2024.pkl')

In [ ]:
df_mapas_nagoya_2024 = pd.read_pickle('df_mapas_nagoya_dec_2024.pkl')

In [ ]:
df_mapas_nagoya_2024

,DATETIME,TECMAP
0,2024-12-01 00:00:00,"[[18.593109130859375, 19.42150115966797, 19.42..."
1,2024-12-01 00:05:00,"[[18.222612380981445, 18.222612380981445, 18.2..."
2,2024-12-01 00:10:00,"[[18.21422576904297, 18.21422576904297, 18.214..."
3,2024-12-01 00:15:00,"[[18.284513473510742, 18.284513473510742, 18.0..."
4,2024-12-01 00:20:00,"[[18.328298568725586, 18.328298568725586, 18.0..."
...,...,...
8347,2024-12-30 23:35:00,"[[26.173906326293945, 27.259140014648438, 27.2..."
8348,2024-12-30 23:40:00,"[[27.0389347076416, 27.0389347076416, 27.03893..."
8349,2024-12-30 23:45:00,"[[26.902502059936523, 26.72080421447754, 25.94..."
8350,2024-12-30 23:50:00,"[[27.88735580444336, 27.88735580444336, 27.887..."


## 12/18/2024 - 18h:20

In [ ]:
import pandas as pd

data = '2024-12-18'
horario = '18:20:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_nagoya_2024[df_mapas_nagoya_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_nagoya_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_nagoya_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_nagoya_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
4828,2024-12-18 18:20:00,"[[18.751976013183594, 18.751976013183594, 18.8..."


In [ ]:
mapas_nagoya = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_nagoya = []
for i in range(len(mapas_nagoya)):
    np_mapas_nagoya.append(mapas_nagoya[i])
np_mapas_nagoya = np.array(np_mapas_nagoya)

In [ ]:
np_mapas_nagoya

array([[[18.75197601, 18.75197601, 18.81381989, ...,         nan,
                 nan,         nan],
        [18.75197601, 18.75197601, 18.81381989, ...,         nan,
                 nan,         nan],
        [18.25943375, 18.13648224, 18.13648224, ...,         nan,
                 nan,         nan],
        ...,
        [38.79235458, 39.10980225, 39.51619339, ..., 13.45065308,
         13.1072731 , 12.96411705],
        [38.38473892, 38.68589783, 39.00465393, ..., 13.27555275,
         12.90675449, 12.72219276],
        [38.23902512, 38.27802277, 38.43577194, ..., 12.968153  ,
         12.60940075, 12.51128292]]])

In [ ]:
np_mapas_nagoya.shape

(1, 242, 221)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


nagoya_tec = nagoya(extent=(-110, 0, -80.4, 40.1), lat_step=0.5, lon_step=0.5)
nagoya_points = nagoya_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(nagoya_points)

[(-23.400000000000006, -46.0, 114, 128), (-23.400000000000006, -45.5, 114, 129), (-22.900000000000006, -46.0, 115, 128), (-22.900000000000006, -45.5, 115, 129)]


In [ ]:
nagoya_points

[(-23.400000000000006, -46.0, 114, 128),
 (-23.400000000000006, -45.5, 114, 129),
 (-22.900000000000006, -46.0, 115, 128),
 (-22.900000000000006, -45.5, 115, 129)]

In [ ]:
nagoya_tec.tec_map = np_mapas_nagoya[0]

print("Os 4 pontos mais próximos são:")
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-23.400000000000006°, -46.0°) - Índices: [114, 128] - Valor TEC: 81.21156311035156
Coordenadas: (-23.400000000000006°, -45.5°) - Índices: [114, 129] - Valor TEC: 81.21156311035156
Coordenadas: (-22.900000000000006°, -46.0°) - Índices: [115, 128] - Valor TEC: 81.97798919677734
Coordenadas: (-22.900000000000006°, -45.5°) - Índices: [115, 129] - Valor TEC: 82.66211700439453

Valor TEC interpolado para (-23.21°, -45.51°): 81.7576


## 12/18/2024 - 18:10

In [ ]:
import pandas as pd

data = '2024-12-18'
horario = '18:10:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_nagoya_2024[df_mapas_nagoya_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_nagoya_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_nagoya_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_nagoya_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
4826,2024-12-18 18:10:00,"[[18.412939071655273, 17.651090621948242, 17.6..."


In [ ]:
mapas_nagoya = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_nagoya = []
for i in range(len(mapas_nagoya)):
    np_mapas_nagoya.append(mapas_nagoya[i])
np_mapas_nagoya = np.array(np_mapas_nagoya)

In [ ]:
np_mapas_nagoya

array([[[18.41293907, 17.65109062, 17.65109062, ...,         nan,
                 nan,         nan],
        [18.53962135, 17.62540054, 17.62540054, ...,         nan,
                 nan,         nan],
        [17.62013054, 17.62013054, 17.62540054, ...,         nan,
                 nan,         nan],
        ...,
        [36.95772552, 37.32144928, 37.6816864 , ..., 14.07102394,
         13.53792286, 13.37452602],
        [37.09318542, 37.28800964, 37.65478134, ..., 13.818923  ,
         13.44021893, 13.28282356],
        [36.133461  , 36.14941025, 36.71422577, ..., 13.40729332,
         13.00967979, 13.02010918]]])

In [ ]:
np_mapas_nagoya.shape

(1, 242, 221)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


nagoya_tec = nagoya(extent=(-110, 0, -80.4, 40.1), lat_step=0.5, lon_step=0.5)
nagoya_points = nagoya_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(nagoya_points)

[(-23.400000000000006, -46.0, 114, 128), (-23.400000000000006, -45.5, 114, 129), (-22.900000000000006, -46.0, 115, 128), (-22.900000000000006, -45.5, 115, 129)]


In [ ]:
nagoya_points

[(-23.400000000000006, -46.0, 114, 128),
 (-23.400000000000006, -45.5, 114, 129),
 (-22.900000000000006, -46.0, 115, 128),
 (-22.900000000000006, -45.5, 115, 129)]

In [ ]:
nagoya_tec.tec_map = np_mapas_nagoya[0]

print("Os 4 pontos mais próximos são:")
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-23.400000000000006°, -46.0°) - Índices: [114, 128] - Valor TEC: 82.88597869873047
Coordenadas: (-23.400000000000006°, -45.5°) - Índices: [114, 129] - Valor TEC: 83.61744689941406
Coordenadas: (-22.900000000000006°, -46.0°) - Índices: [115, 128] - Valor TEC: 84.93486022949219
Coordenadas: (-22.900000000000006°, -45.5°) - Índices: [115, 129] - Valor TEC: 85.56057739257812

Valor TEC interpolado para (-23.21°, -45.51°): 84.3420


## 12/18/2024 - 18:30

In [ ]:
import pandas as pd

data = '2024-12-18'
horario = '18:30:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_nagoya_2024[df_mapas_nagoya_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_nagoya_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_nagoya_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_nagoya_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
4830,2024-12-18 18:30:00,"[[21.299997329711914, 21.299997329711914, 21.2..."


In [ ]:
mapas_nagoya = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_nagoya = []
for i in range(len(mapas_nagoya)):
    np_mapas_nagoya.append(mapas_nagoya[i])
np_mapas_nagoya = np.array(np_mapas_nagoya)

In [ ]:
np_mapas_nagoya

array([[[21.29999733, 21.29999733, 21.29999733, ...,         nan,
                 nan,         nan],
        [21.29999733, 21.29999733, 21.29999733, ...,         nan,
                 nan,         nan],
        [21.28308678, 21.28308678, 21.28308678, ...,         nan,
                 nan,         nan],
        ...,
        [39.65393066, 40.11976624, 40.50592804, ..., 13.14062977,
         12.80635357, 12.63247967],
        [39.26050949, 39.7519455 , 40.19271851, ..., 12.65815067,
         12.54124069, 12.46903992],
        [39.23273468, 39.40169144, 39.72786331, ..., 12.41631317,
         12.32884121, 12.25786972]]])

In [ ]:
np_mapas_nagoya.shape

(1, 242, 221)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


nagoya_tec = nagoya(extent=(-110, 0, -80.4, 40.1), lat_step=0.5, lon_step=0.5)
nagoya_points = nagoya_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(nagoya_points)

[(-23.400000000000006, -46.0, 114, 128), (-23.400000000000006, -45.5, 114, 129), (-22.900000000000006, -46.0, 115, 128), (-22.900000000000006, -45.5, 115, 129)]


In [ ]:
nagoya_points

[(-23.400000000000006, -46.0, 114, 128),
 (-23.400000000000006, -45.5, 114, 129),
 (-22.900000000000006, -46.0, 115, 128),
 (-22.900000000000006, -45.5, 115, 129)]

In [ ]:
nagoya_tec.tec_map = np_mapas_nagoya[0]

print("Os 4 pontos mais próximos são:")
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-23.400000000000006°, -46.0°) - Índices: [114, 128] - Valor TEC: 82.04778289794922
Coordenadas: (-23.400000000000006°, -45.5°) - Índices: [114, 129] - Valor TEC: 80.9701156616211
Coordenadas: (-22.900000000000006°, -46.0°) - Índices: [115, 128] - Valor TEC: 83.57044982910156
Coordenadas: (-22.900000000000006°, -45.5°) - Índices: [115, 129] - Valor TEC: 81.77594757080078

Valor TEC interpolado para (-23.21°, -45.51°): 81.3033


## 12/18/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-18'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_nagoya_2024[df_mapas_nagoya_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_nagoya_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_nagoya_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_nagoya_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
4824,2024-12-18 18:00:00,"[[17.569849014282227, 17.569849014282227, 17.6..."


In [ ]:
mapas_nagoya = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_nagoya = []
for i in range(len(mapas_nagoya)):
    np_mapas_nagoya.append(mapas_nagoya[i])
np_mapas_nagoya = np.array(np_mapas_nagoya)

In [ ]:
np_mapas_nagoya

array([[[17.56984901, 17.56984901, 17.63764763, ...,         nan,
                 nan,         nan],
        [17.83648491, 17.83648491, 17.8645153 , ...,         nan,
                 nan,         nan],
        [16.60485077, 16.60485077, 16.60485077, ...,         nan,
                 nan,         nan],
        ...,
        [35.25151062, 35.38899231, 35.7537117 , ..., 14.51318359,
         13.86343384, 13.80283642],
        [34.86306763, 34.9417305 , 34.75822449, ..., 14.43698978,
         13.88199997, 13.79667377],
        [34.32035065, 34.41825104, 34.44206238, ..., 14.2071352 ,
         13.66319656, 13.62114334]]])

In [ ]:
np_mapas_nagoya.shape

(1, 242, 221)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


nagoya_tec = nagoya(extent=(-110, 0, -80.4, 40.1), lat_step=0.5, lon_step=0.5)
nagoya_points = nagoya_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(nagoya_points)

[(-23.400000000000006, -46.0, 114, 128), (-23.400000000000006, -45.5, 114, 129), (-22.900000000000006, -46.0, 115, 128), (-22.900000000000006, -45.5, 115, 129)]


In [ ]:
nagoya_points

[(-23.400000000000006, -46.0, 114, 128),
 (-23.400000000000006, -45.5, 114, 129),
 (-22.900000000000006, -46.0, 115, 128),
 (-22.900000000000006, -45.5, 115, 129)]

In [ ]:
nagoya_tec.tec_map = np_mapas_nagoya[0]

print("Os 4 pontos mais próximos são:")
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-23.400000000000006°, -46.0°) - Índices: [114, 128] - Valor TEC: 82.92593383789062
Coordenadas: (-23.400000000000006°, -45.5°) - Índices: [114, 129] - Valor TEC: 81.68982696533203
Coordenadas: (-22.900000000000006°, -46.0°) - Índices: [115, 128] - Valor TEC: 84.24528503417969
Coordenadas: (-22.900000000000006°, -45.5°) - Índices: [115, 129] - Valor TEC: 82.12318420410156

Valor TEC interpolado para (-23.21°, -45.51°): 81.8860


## 12/22/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-22'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_nagoya_2024[df_mapas_nagoya_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_nagoya_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_nagoya_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_nagoya_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
5976,2024-12-22 18:00:00,"[[14.305987358093262, 14.235016822814941, 15.1..."


In [ ]:
mapas_nagoya = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_nagoya = []
for i in range(len(mapas_nagoya)):
    np_mapas_nagoya.append(mapas_nagoya[i])
np_mapas_nagoya = np.array(np_mapas_nagoya)

In [ ]:
np_mapas_nagoya

array([[[14.30598736, 14.23501682, 15.19717693, ...,         nan,
                 nan,         nan],
        [13.24124527, 13.24124527, 13.24124527, ...,         nan,
                 nan,         nan],
        [13.24124527, 13.24124527, 13.24124527, ...,         nan,
                 nan,         nan],
        ...,
        [43.63508224, 43.97467422, 44.23501205, ..., 20.72278595,
         20.15914345, 19.78045845],
        [43.53390503, 43.86975861, 44.0747261 , ..., 20.29078865,
         19.64533615, 19.32730103],
        [43.54644394, 43.77022934, 43.95893478, ..., 19.72059822,
         19.0694046 , 18.65716362]]])

In [ ]:
np_mapas_nagoya.shape

(1, 242, 221)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


nagoya_tec = nagoya(extent=(-110, 0, -80.4, 40.1), lat_step=0.5, lon_step=0.5)
nagoya_points = nagoya_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(nagoya_points)

[(-23.400000000000006, -46.0, 114, 128), (-23.400000000000006, -45.5, 114, 129), (-22.900000000000006, -46.0, 115, 128), (-22.900000000000006, -45.5, 115, 129)]


In [ ]:
nagoya_points

[(-23.400000000000006, -46.0, 114, 128),
 (-23.400000000000006, -45.5, 114, 129),
 (-22.900000000000006, -46.0, 115, 128),
 (-22.900000000000006, -45.5, 115, 129)]

In [ ]:
nagoya_tec.tec_map = np_mapas_nagoya[0]

print("Os 4 pontos mais próximos são:")
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-23.400000000000006°, -46.0°) - Índices: [114, 128] - Valor TEC: 80.86611938476562
Coordenadas: (-23.400000000000006°, -45.5°) - Índices: [114, 129] - Valor TEC: 80.97747802734375
Coordenadas: (-22.900000000000006°, -46.0°) - Índices: [115, 128] - Valor TEC: 80.97673797607422
Coordenadas: (-22.900000000000006°, -45.5°) - Índices: [115, 129] - Valor TEC: 81.30828094482422

Valor TEC interpolado para (-23.21°, -45.51°): 81.0993


## 12/19/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-19'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_nagoya_2024[df_mapas_nagoya_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_nagoya_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_nagoya_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_nagoya_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
5112,2024-12-19 18:00:00,"[[12.639724731445312, 11.565997123718262, 11.5..."


In [ ]:
mapas_nagoya = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_nagoya = []
for i in range(len(mapas_nagoya)):
    np_mapas_nagoya.append(mapas_nagoya[i])
np_mapas_nagoya = np.array(np_mapas_nagoya)

In [ ]:
np_mapas_nagoya

array([[[12.63972473, 11.56599712, 11.56599712, ...,         nan,
                 nan,         nan],
        [11.71732426, 10.61837482, 10.61837482, ...,         nan,
                 nan,         nan],
        [14.18716908, 11.54071808, 10.25730038, ...,         nan,
                 nan,         nan],
        ...,
        [34.01515198, 34.48353958, 34.89712906, ..., 11.77007103,
         11.27845764, 11.27845764],
        [33.72997284, 33.06239319, 33.53577042, ..., 11.66871548,
         11.30887318, 11.18506527],
        [33.0634613 , 32.32966614, 32.78850937, ..., 11.32422161,
         11.1510849 , 11.04978752]]])

In [ ]:
np_mapas_nagoya.shape

(1, 242, 221)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


nagoya_tec = nagoya(extent=(-110, 0, -80.4, 40.1), lat_step=0.5, lon_step=0.5)
nagoya_points = nagoya_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(nagoya_points)

[(-23.400000000000006, -46.0, 114, 128), (-23.400000000000006, -45.5, 114, 129), (-22.900000000000006, -46.0, 115, 128), (-22.900000000000006, -45.5, 115, 129)]


In [ ]:
nagoya_points

[(-23.400000000000006, -46.0, 114, 128),
 (-23.400000000000006, -45.5, 114, 129),
 (-22.900000000000006, -46.0, 115, 128),
 (-22.900000000000006, -45.5, 115, 129)]

In [ ]:
nagoya_tec.tec_map = np_mapas_nagoya[0]

print("Os 4 pontos mais próximos são:")
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-23.400000000000006°, -46.0°) - Índices: [114, 128] - Valor TEC: 76.62113952636719
Coordenadas: (-23.400000000000006°, -45.5°) - Índices: [114, 129] - Valor TEC: 76.78427124023438
Coordenadas: (-22.900000000000006°, -46.0°) - Índices: [115, 128] - Valor TEC: 75.99300384521484
Coordenadas: (-22.900000000000006°, -45.5°) - Índices: [115, 129] - Valor TEC: 75.99300384521484

Valor TEC interpolado para (-23.21°, -45.51°): 76.4816


## 12/01/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-01'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_nagoya_2024[df_mapas_nagoya_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_nagoya_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_nagoya_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_nagoya_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
216,2024-12-01 18:00:00,"[[5.463457107543945, 5.463457107543945, 5.4634..."


In [ ]:
mapas_nagoya = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_nagoya = []
for i in range(len(mapas_nagoya)):
    np_mapas_nagoya.append(mapas_nagoya[i])
np_mapas_nagoya = np.array(np_mapas_nagoya)

In [ ]:
np_mapas_nagoya

array([[[ 5.46345711,  5.46345711,  5.46345711, ...,         nan,
                 nan,         nan],
        [ 9.9241066 , 10.95889568, 10.95889568, ...,         nan,
                 nan,         nan],
        [14.80212975, 14.80212975, 14.12398911, ...,         nan,
                 nan,         nan],
        ...,
        [43.88991547, 44.48843002, 44.89578247, ..., 21.9361248 ,
         21.8728199 , 21.43001556],
        [44.12051773, 44.59725952, 44.77046967, ..., 21.9361248 ,
         21.76833344, 21.42024994],
        [44.22133255, 44.64913177, 44.84080887, ..., 21.9361248 ,
         21.67697716, 21.29607773]]])

In [ ]:
np_mapas_nagoya.shape

(1, 242, 221)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


nagoya_tec = nagoya(extent=(-110, 0, -80.4, 40.1), lat_step=0.5, lon_step=0.5)
nagoya_points = nagoya_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(nagoya_points)

[(-23.400000000000006, -46.0, 114, 128), (-23.400000000000006, -45.5, 114, 129), (-22.900000000000006, -46.0, 115, 128), (-22.900000000000006, -45.5, 115, 129)]


In [ ]:
nagoya_points

[(-23.400000000000006, -46.0, 114, 128),
 (-23.400000000000006, -45.5, 114, 129),
 (-22.900000000000006, -46.0, 115, 128),
 (-22.900000000000006, -45.5, 115, 129)]

In [ ]:
nagoya_tec.tec_map = np_mapas_nagoya[0]

print("Os 4 pontos mais próximos são:")
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-23.400000000000006°, -46.0°) - Índices: [114, 128] - Valor TEC: 78.74305725097656
Coordenadas: (-23.400000000000006°, -45.5°) - Índices: [114, 129] - Valor TEC: 78.46280670166016
Coordenadas: (-22.900000000000006°, -46.0°) - Índices: [115, 128] - Valor TEC: 78.47337341308594
Coordenadas: (-22.900000000000006°, -45.5°) - Índices: [115, 129] - Valor TEC: 78.26663970947266

Valor TEC interpolado para (-23.21°, -45.51°): 78.3933


# EMBRACE

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
%cd '/content'

/content


In [ ]:
#df_mapas_embrace_sept_05_2024_beyond.pkl
!gdown '1LkY0IeFfdnahoH81OKYW-2xmXHgjFPhH'

Downloading...
From (original): https://drive.google.com/uc?id=1LkY0IeFfdnahoH81OKYW-2xmXHgjFPhH
From (redirected): https://drive.google.com/uc?id=1LkY0IeFfdnahoH81OKYW-2xmXHgjFPhH&confirm=t&uuid=5a880249-1e7d-447e-a523-012f6a3c890d
To: /content/df_mapas_embrace_sept_05_2024_beyond.pkl
100% 157M/157M [00:01<00:00, 124MB/s]


In [ ]:
df_mapas_embrace_2024 = pd.read_pickle('/content/df_mapas_embrace_sept_05_2024_beyond.pkl')

In [ ]:
df_mapas_embrace_2024

,TECMAP
DATETIME,
2024-09-05 00:00:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
2024-09-05 00:10:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
2024-09-05 00:20:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
2024-09-05 00:30:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
2024-09-05 00:40:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
...,...
2024-12-30 23:10:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
2024-12-30 23:20:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
2024-12-30 23:30:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


## 12/18/2024 - 18:20

In [ ]:
import pandas as pd

data = '2024-12-18'
horario = '18:20:00'
timestamp_procurado = f'{data} {horario}'

if df_mapas_embrace_2024.index.name == 'DATETIME':
    resultado = df_mapas_embrace_2024.loc[timestamp_procurado:timestamp_procurado]
else:
    if 'DATETIME' in df_mapas_embrace_2024.columns:
        df_mapas_embrace_2024 = df_mapas_embrace_2024.set_index('DATETIME')
        resultado = df_mapas_embrace_2024.loc[timestamp_procurado:timestamp_procurado]
    else:
        print("Coluna DATETIME não encontrada!")
        print("Colunas disponíveis:", df_mapas_embrace_2024.columns.tolist())

if resultado.empty:
    if not isinstance(df_mapas_embrace_2024.index, pd.DatetimeIndex):
        df_mapas_embrace_2024.index = pd.to_datetime(df_mapas_embrace_2024.index)

    timestamp_dt = pd.to_datetime(timestamp_procurado)

    diferenca = abs((df_mapas_embrace_2024.index - timestamp_dt).total_seconds())
    idx_mais_proximo = diferenca.argmin()

    resultado = df_mapas_embrace_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_embrace_2024.index[idx_mais_proximo]}")

In [ ]:
resultado

,TECMAP
DATETIME,
2024-12-18 18:20:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


In [ ]:
mapas_embrace = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_embrace = []
for i in range(len(mapas_embrace)):
    np_mapas_embrace.append(mapas_embrace[i])
np_mapas_embrace = np.array(np_mapas_embrace)

In [ ]:
np_mapas_embrace

array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, 39., nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]])

In [ ]:
np_mapas_embrace.shape

(1, 41, 25)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(embrace_points)

[(-24, -47.5, 18, 17), (-24, -45.0, 18, 18), (-22, -47.5, 19, 17), (-22, -45.0, 19, 18)]


In [ ]:
embrace_points

[(-24, -47.5, 18, 17),
 (-24, -45.0, 18, 18),
 (-22, -47.5, 19, 17),
 (-22, -45.0, 19, 18)]

In [ ]:
embrace_tec.tec_map = np_mapas_embrace[0]

print("Os 4 pontos mais próximos são:")
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-24°, -47.5°) - Índices: [18, 17] - Valor TEC: 74.0
Coordenadas: (-24°, -45.0°) - Índices: [18, 18] - Valor TEC: 80.0
Coordenadas: (-22°, -47.5°) - Índices: [19, 17] - Valor TEC: 80.0
Coordenadas: (-22°, -45.0°) - Índices: [19, 18] - Valor TEC: 86.0

Valor TEC interpolado para (-23.21°, -45.51°): 81.1460


## 12/18/2024 - 18:10

In [ ]:
import pandas as pd

data = '2024-12-18'
horario = '18:10:00'
timestamp_procurado = f'{data} {horario}'

if df_mapas_embrace_2024.index.name == 'DATETIME':
    resultado = df_mapas_embrace_2024.loc[timestamp_procurado:timestamp_procurado]
else:
    if 'DATETIME' in df_mapas_embrace_2024.columns:
        df_mapas_embrace_2024 = df_mapas_embrace_2024.set_index('DATETIME')
        resultado = df_mapas_embrace_2024.loc[timestamp_procurado:timestamp_procurado]
    else:
        print("Coluna DATETIME não encontrada!")
        print("Colunas disponíveis:", df_mapas_embrace_2024.columns.tolist())

if resultado.empty:
    if not isinstance(df_mapas_embrace_2024.index, pd.DatetimeIndex):
        df_mapas_embrace_2024.index = pd.to_datetime(df_mapas_embrace_2024.index)

    timestamp_dt = pd.to_datetime(timestamp_procurado)

    diferenca = abs((df_mapas_embrace_2024.index - timestamp_dt).total_seconds())
    idx_mais_proximo = diferenca.argmin()

    resultado = df_mapas_embrace_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_embrace_2024.index[idx_mais_proximo]}")

In [ ]:
resultado

,TECMAP
DATETIME,
2024-12-18 18:10:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


In [ ]:
mapas_embrace = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_embrace = []
for i in range(len(mapas_embrace)):
    np_mapas_embrace.append(mapas_embrace[i])
np_mapas_embrace = np.array(np_mapas_embrace)

In [ ]:
np_mapas_embrace

array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, 37., nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]])

In [ ]:
np_mapas_embrace.shape

(1, 41, 25)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(embrace_points)

[(-24, -47.5, 18, 17), (-24, -45.0, 18, 18), (-22, -47.5, 19, 17), (-22, -45.0, 19, 18)]


In [ ]:
embrace_points

[(-24, -47.5, 18, 17),
 (-24, -45.0, 18, 18),
 (-22, -47.5, 19, 17),
 (-22, -45.0, 19, 18)]

In [ ]:
embrace_tec.tec_map = np_mapas_embrace[0]

print("Os 4 pontos mais próximos são:")
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-24°, -47.5°) - Índices: [18, 17] - Valor TEC: 78.0
Coordenadas: (-24°, -45.0°) - Índices: [18, 18] - Valor TEC: 79.0
Coordenadas: (-22°, -47.5°) - Índices: [19, 17] - Valor TEC: 86.0
Coordenadas: (-22°, -45.0°) - Índices: [19, 18] - Valor TEC: 83.0

Valor TEC interpolado para (-23.21°, -45.51°): 80.6983


## 12/18/2024 - 18:30

In [ ]:
import pandas as pd

data = '2024-12-18'
horario = '18:30:00'
timestamp_procurado = f'{data} {horario}'

if df_mapas_embrace_2024.index.name == 'DATETIME':
    resultado = df_mapas_embrace_2024.loc[timestamp_procurado:timestamp_procurado]
else:
    if 'DATETIME' in df_mapas_embrace_2024.columns:
        df_mapas_embrace_2024 = df_mapas_embrace_2024.set_index('DATETIME')
        resultado = df_mapas_embrace_2024.loc[timestamp_procurado:timestamp_procurado]
    else:
        print("Coluna DATETIME não encontrada!")
        print("Colunas disponíveis:", df_mapas_embrace_2024.columns.tolist())

if resultado.empty:
    if not isinstance(df_mapas_embrace_2024.index, pd.DatetimeIndex):
        df_mapas_embrace_2024.index = pd.to_datetime(df_mapas_embrace_2024.index)

    timestamp_dt = pd.to_datetime(timestamp_procurado)

    diferenca = abs((df_mapas_embrace_2024.index - timestamp_dt).total_seconds())
    idx_mais_proximo = diferenca.argmin()

    resultado = df_mapas_embrace_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_embrace_2024.index[idx_mais_proximo]}")

In [ ]:
resultado

,TECMAP
DATETIME,
2024-12-18 18:30:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


In [ ]:
mapas_embrace = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_embrace = []
for i in range(len(mapas_embrace)):
    np_mapas_embrace.append(mapas_embrace[i])
np_mapas_embrace = np.array(np_mapas_embrace)

In [ ]:
np_mapas_embrace

array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, 39., nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]])

In [ ]:
np_mapas_embrace.shape

(1, 41, 25)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(embrace_points)

[(-24, -47.5, 18, 17), (-24, -45.0, 18, 18), (-22, -47.5, 19, 17), (-22, -45.0, 19, 18)]


In [ ]:
embrace_points

[(-24, -47.5, 18, 17),
 (-24, -45.0, 18, 18),
 (-22, -47.5, 19, 17),
 (-22, -45.0, 19, 18)]

In [ ]:
embrace_tec.tec_map = np_mapas_embrace[0]

print("Os 4 pontos mais próximos são:")
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-24°, -47.5°) - Índices: [18, 17] - Valor TEC: 77.0
Coordenadas: (-24°, -45.0°) - Índices: [18, 18] - Valor TEC: 79.0
Coordenadas: (-22°, -47.5°) - Índices: [19, 17] - Valor TEC: 92.0
Coordenadas: (-22°, -45.0°) - Índices: [19, 18] - Valor TEC: 88.0

Valor TEC interpolado para (-23.21°, -45.51°): 82.6305


## 12/18/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-18'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

if df_mapas_embrace_2024.index.name == 'DATETIME':
    resultado = df_mapas_embrace_2024.loc[timestamp_procurado:timestamp_procurado]
else:
    if 'DATETIME' in df_mapas_embrace_2024.columns:
        df_mapas_embrace_2024 = df_mapas_embrace_2024.set_index('DATETIME')
        resultado = df_mapas_embrace_2024.loc[timestamp_procurado:timestamp_procurado]
    else:
        print("Coluna DATETIME não encontrada!")
        print("Colunas disponíveis:", df_mapas_embrace_2024.columns.tolist())

if resultado.empty:
    if not isinstance(df_mapas_embrace_2024.index, pd.DatetimeIndex):
        df_mapas_embrace_2024.index = pd.to_datetime(df_mapas_embrace_2024.index)

    timestamp_dt = pd.to_datetime(timestamp_procurado)

    diferenca = abs((df_mapas_embrace_2024.index - timestamp_dt).total_seconds())
    idx_mais_proximo = diferenca.argmin()

    resultado = df_mapas_embrace_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_embrace_2024.index[idx_mais_proximo]}")

In [ ]:
resultado

,TECMAP
DATETIME,
2024-12-18 18:00:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


In [ ]:
mapas_embrace = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_embrace = []
for i in range(len(mapas_embrace)):
    np_mapas_embrace.append(mapas_embrace[i])
np_mapas_embrace = np.array(np_mapas_embrace)

In [ ]:
np_mapas_embrace

array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, 36., nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]])

In [ ]:
np_mapas_embrace.shape

(1, 41, 25)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(embrace_points)

[(-24, -47.5, 18, 17), (-24, -45.0, 18, 18), (-22, -47.5, 19, 17), (-22, -45.0, 19, 18)]


In [ ]:
embrace_points

[(-24, -47.5, 18, 17),
 (-24, -45.0, 18, 18),
 (-22, -47.5, 19, 17),
 (-22, -45.0, 19, 18)]

In [ ]:
embrace_tec.tec_map = np_mapas_embrace[0]

print("Os 4 pontos mais próximos são:")
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-24°, -47.5°) - Índices: [18, 17] - Valor TEC: 79.0
Coordenadas: (-24°, -45.0°) - Índices: [18, 18] - Valor TEC: 77.0
Coordenadas: (-22°, -47.5°) - Índices: [19, 17] - Valor TEC: 81.0
Coordenadas: (-22°, -45.0°) - Índices: [19, 18] - Valor TEC: 86.0

Valor TEC interpolado para (-23.21°, -45.51°): 80.3989


## 12/22/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-22'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

if df_mapas_embrace_2024.index.name == 'DATETIME':
    resultado = df_mapas_embrace_2024.loc[timestamp_procurado:timestamp_procurado]
else:
    if 'DATETIME' in df_mapas_embrace_2024.columns:
        df_mapas_embrace_2024 = df_mapas_embrace_2024.set_index('DATETIME')
        resultado = df_mapas_embrace_2024.loc[timestamp_procurado:timestamp_procurado]
    else:
        print("Coluna DATETIME não encontrada!")
        print("Colunas disponíveis:", df_mapas_embrace_2024.columns.tolist())

if resultado.empty:
    if not isinstance(df_mapas_embrace_2024.index, pd.DatetimeIndex):
        df_mapas_embrace_2024.index = pd.to_datetime(df_mapas_embrace_2024.index)

    timestamp_dt = pd.to_datetime(timestamp_procurado)

    diferenca = abs((df_mapas_embrace_2024.index - timestamp_dt).total_seconds())
    idx_mais_proximo = diferenca.argmin()

    resultado = df_mapas_embrace_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_embrace_2024.index[idx_mais_proximo]}")

In [ ]:
resultado

,TECMAP
DATETIME,
2024-12-22 18:00:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


In [ ]:
mapas_embrace = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_embrace = []
for i in range(len(mapas_embrace)):
    np_mapas_embrace.append(mapas_embrace[i])
np_mapas_embrace = np.array(np_mapas_embrace)

In [ ]:
np_mapas_embrace

array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, 52., nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]])

In [ ]:
np_mapas_embrace.shape

(1, 41, 25)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(embrace_points)

[(-24, -47.5, 18, 17), (-24, -45.0, 18, 18), (-22, -47.5, 19, 17), (-22, -45.0, 19, 18)]


In [ ]:
embrace_points

[(-24, -47.5, 18, 17),
 (-24, -45.0, 18, 18),
 (-22, -47.5, 19, 17),
 (-22, -45.0, 19, 18)]

In [ ]:
embrace_tec.tec_map = np_mapas_embrace[0]

print("Os 4 pontos mais próximos são:")
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-24°, -47.5°) - Índices: [18, 17] - Valor TEC: 79.0
Coordenadas: (-24°, -45.0°) - Índices: [18, 18] - Valor TEC: 78.0
Coordenadas: (-22°, -47.5°) - Índices: [19, 17] - Valor TEC: 75.0
Coordenadas: (-22°, -45.0°) - Índices: [19, 18] - Valor TEC: 79.0

Valor TEC interpolado para (-23.21°, -45.51°): 78.1961


## 12/19/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-19'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

if df_mapas_embrace_2024.index.name == 'DATETIME':
    resultado = df_mapas_embrace_2024.loc[timestamp_procurado:timestamp_procurado]
else:
    if 'DATETIME' in df_mapas_embrace_2024.columns:
        df_mapas_embrace_2024 = df_mapas_embrace_2024.set_index('DATETIME')
        resultado = df_mapas_embrace_2024.loc[timestamp_procurado:timestamp_procurado]
    else:
        print("Coluna DATETIME não encontrada!")
        print("Colunas disponíveis:", df_mapas_embrace_2024.columns.tolist())

if resultado.empty:
    if not isinstance(df_mapas_embrace_2024.index, pd.DatetimeIndex):
        df_mapas_embrace_2024.index = pd.to_datetime(df_mapas_embrace_2024.index)

    timestamp_dt = pd.to_datetime(timestamp_procurado)

    diferenca = abs((df_mapas_embrace_2024.index - timestamp_dt).total_seconds())
    idx_mais_proximo = diferenca.argmin()

    resultado = df_mapas_embrace_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_embrace_2024.index[idx_mais_proximo]}")

In [ ]:
resultado

,TECMAP
DATETIME,
2024-12-19 18:00:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


In [ ]:
mapas_embrace = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_embrace = []
for i in range(len(mapas_embrace)):
    np_mapas_embrace.append(mapas_embrace[i])
np_mapas_embrace = np.array(np_mapas_embrace)

In [ ]:
np_mapas_embrace

array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, 41., nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]])

In [ ]:
np_mapas_embrace.shape

(1, 41, 25)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(embrace_points)

[(-24, -47.5, 18, 17), (-24, -45.0, 18, 18), (-22, -47.5, 19, 17), (-22, -45.0, 19, 18)]


In [ ]:
embrace_points

[(-24, -47.5, 18, 17),
 (-24, -45.0, 18, 18),
 (-22, -47.5, 19, 17),
 (-22, -45.0, 19, 18)]

In [ ]:
embrace_tec.tec_map = np_mapas_embrace[0]

print("Os 4 pontos mais próximos são:")
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-24°, -47.5°) - Índices: [18, 17] - Valor TEC: 79.0
Coordenadas: (-24°, -45.0°) - Índices: [18, 18] - Valor TEC: 76.0
Coordenadas: (-22°, -47.5°) - Índices: [19, 17] - Valor TEC: 71.0
Coordenadas: (-22°, -45.0°) - Índices: [19, 18] - Valor TEC: 80.0

Valor TEC interpolado para (-23.21°, -45.51°): 77.2250


## 12/01/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-01'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

if df_mapas_embrace_2024.index.name == 'DATETIME':
    resultado = df_mapas_embrace_2024.loc[timestamp_procurado:timestamp_procurado]
else:
    if 'DATETIME' in df_mapas_embrace_2024.columns:
        df_mapas_embrace_2024 = df_mapas_embrace_2024.set_index('DATETIME')
        resultado = df_mapas_embrace_2024.loc[timestamp_procurado:timestamp_procurado]
    else:
        print("Coluna DATETIME não encontrada!")
        print("Colunas disponíveis:", df_mapas_embrace_2024.columns.tolist())

if resultado.empty:
    if not isinstance(df_mapas_embrace_2024.index, pd.DatetimeIndex):
        df_mapas_embrace_2024.index = pd.to_datetime(df_mapas_embrace_2024.index)

    timestamp_dt = pd.to_datetime(timestamp_procurado)

    diferenca = abs((df_mapas_embrace_2024.index - timestamp_dt).total_seconds())
    idx_mais_proximo = diferenca.argmin()

    resultado = df_mapas_embrace_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_embrace_2024.index[idx_mais_proximo]}")

In [ ]:
resultado

,TECMAP
DATETIME,
2024-12-01 18:00:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


In [ ]:
mapas_embrace = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_embrace = []
for i in range(len(mapas_embrace)):
    np_mapas_embrace.append(mapas_embrace[i])
np_mapas_embrace = np.array(np_mapas_embrace)

In [ ]:
np_mapas_embrace

array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, 46., nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]])

In [ ]:
np_mapas_embrace.shape

(1, 41, 25)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(embrace_points)

[(-24, -47.5, 18, 17), (-24, -45.0, 18, 18), (-22, -47.5, 19, 17), (-22, -45.0, 19, 18)]


In [ ]:
embrace_points

[(-24, -47.5, 18, 17),
 (-24, -45.0, 18, 18),
 (-22, -47.5, 19, 17),
 (-22, -45.0, 19, 18)]

In [ ]:
embrace_tec.tec_map = np_mapas_embrace[0]

print("Os 4 pontos mais próximos são:")
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-24°, -47.5°) - Índices: [18, 17] - Valor TEC: 76.0
Coordenadas: (-24°, -45.0°) - Índices: [18, 18] - Valor TEC: 78.0
Coordenadas: (-22°, -47.5°) - Índices: [19, 17] - Valor TEC: 79.0
Coordenadas: (-22°, -45.0°) - Índices: [19, 18] - Valor TEC: 80.0

Valor TEC interpolado para (-23.21°, -45.51°): 78.4626


# MAGGIA

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#df_mapas_maggia_sept_05_2024_beyond.pkl
!gdown '1Qi7Xm93lhuwGWQnr3N-gquhzzUxegI33'

In [ ]:
df_mapas_maggia_2024 = pd.read_pickle('/content/df_mapas_maggia_sept_05_2024_beyond.pkl')

In [ ]:
df_mapas_maggia_2024

,DATETIME,TECMAP
0,2024-09-05 00:00:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
1,2024-09-05 00:10:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
2,2024-09-05 00:15:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
3,2024-09-05 00:20:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
4,2024-09-05 00:30:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
...,...,...
15619,2024-12-31 23:10:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
15620,2024-12-31 23:20:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
15621,2024-12-31 23:30:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."
15622,2024-12-31 23:40:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


## 12/18/2024 - 18:20

In [ ]:
import pandas as pd

data = '2024-12-18'
horario = '18:20:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_maggia_2024[df_mapas_maggia_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_maggia_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_maggia_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_maggia_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
13718,2024-12-18 18:20:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


In [ ]:
mapas_maggia = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_maggia = []
for i in range(len(mapas_maggia)):
    np_mapas_maggia.append(mapas_maggia[i])
np_mapas_maggia = np.array(np_mapas_maggia)

In [ ]:
np_mapas_maggia

array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [40.14899826, 40.15000153, 40.18000031, ..., 13.84500027,
         13.67500019, 13.12800026],
        [39.97399902, 40.02799988, 40.15399933, ..., 13.75800037,
         13.67599964, 12.39599991],
        [39.73500061, 39.83499908, 40.02500153, ..., 13.10700035,
         12.6239996 , 12.06000042]]])

In [ ]:
np_mapas_maggia.shape

(1, 241, 221)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


maggia_tec = maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(maggia_points)

[(-23.5, -46.0, 113, 128), (-23.5, -45.5, 113, 129), (-23.0, -46.0, 114, 128), (-23.0, -45.5, 114, 129)]


In [ ]:
maggia_points

[(-23.5, -46.0, 113, 128),
 (-23.5, -45.5, 113, 129),
 (-23.0, -46.0, 114, 128),
 (-23.0, -45.5, 114, 129)]

In [ ]:
maggia_tec.tec_map = np_mapas_maggia[0]

print("Os 4 pontos mais próximos são:")
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-23.5°, -46.0°) - Índices: [113, 128] - Valor TEC: 84.3280029296875
Coordenadas: (-23.5°, -45.5°) - Índices: [113, 129] - Valor TEC: 81.5530014038086
Coordenadas: (-23.0°, -46.0°) - Índices: [114, 128] - Valor TEC: 86.61299896240234
Coordenadas: (-23.0°, -45.5°) - Índices: [114, 129] - Valor TEC: 84.23500061035156

Valor TEC interpolado para (-23.21°, -45.51°): 83.1595


## 12/18/2024 - 18:10

In [ ]:
import pandas as pd

data = '2024-12-18'
horario = '18:10:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_maggia_2024[df_mapas_maggia_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_maggia_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_maggia_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_maggia_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
13717,2024-12-18 18:10:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


In [ ]:
mapas_maggia = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_maggia = []
for i in range(len(mapas_maggia)):
    np_mapas_maggia.append(mapas_maggia[i])
np_mapas_maggia = np.array(np_mapas_maggia)

In [ ]:
np_mapas_maggia

array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [36.83499908, 37.06399918, 37.23899841, ..., 12.40799999,
         12.40799999, 12.40799999],
        [36.57699966, 36.82799911, 37.03300095, ..., 12.85900021,
         12.6619997 , 12.54399967],
        [36.34600067, 36.5909996 , 36.81399918, ..., 13.00500011,
         12.82999992, 12.32600021]]])

In [ ]:
np_mapas_maggia.shape

(1, 241, 221)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


maggia_tec = maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(maggia_points)

[(-23.5, -46.0, 113, 128), (-23.5, -45.5, 113, 129), (-23.0, -46.0, 114, 128), (-23.0, -45.5, 114, 129)]


In [ ]:
maggia_points

[(-23.5, -46.0, 113, 128),
 (-23.5, -45.5, 113, 129),
 (-23.0, -46.0, 114, 128),
 (-23.0, -45.5, 114, 129)]

In [ ]:
maggia_tec.tec_map = np_mapas_maggia[0]

print("Os 4 pontos mais próximos são:")
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-23.5°, -46.0°) - Índices: [113, 128] - Valor TEC: 85.65799713134766
Coordenadas: (-23.5°, -45.5°) - Índices: [113, 129] - Valor TEC: 80.13600158691406
Coordenadas: (-23.0°, -46.0°) - Índices: [114, 128] - Valor TEC: 85.75399780273438
Coordenadas: (-23.0°, -45.5°) - Índices: [114, 129] - Valor TEC: 83.74800109863281

Valor TEC interpolado para (-23.21°, -45.51°): 82.3006


## 12/18/2024 - 18:30

In [ ]:
import pandas as pd

data = '2024-12-18'
horario = '18:30:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_maggia_2024[df_mapas_maggia_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_maggia_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_maggia_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_maggia_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
13719,2024-12-18 18:30:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


In [ ]:
mapas_maggia = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_maggia = []
for i in range(len(mapas_maggia)):
    np_mapas_maggia.append(mapas_maggia[i])
np_mapas_maggia = np.array(np_mapas_maggia)

In [ ]:
np_mapas_maggia

array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [41.94900131, 41.94900131, 41.94900131, ..., 11.0880003 ,
         11.0880003 , 11.0880003 ],
        [41.94400024, 41.94400024, 41.94499969, ..., 11.0880003 ,
         11.0880003 , 11.0880003 ],
        [41.89500046, 41.89799881, 41.90100098, ..., 11.36600018,
         11.25800037, 11.1960001 ]]])

In [ ]:
np_mapas_maggia.shape

(1, 241, 221)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


maggia_tec = maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(maggia_points)

[(-23.5, -46.0, 113, 128), (-23.5, -45.5, 113, 129), (-23.0, -46.0, 114, 128), (-23.0, -45.5, 114, 129)]


In [ ]:
maggia_points

[(-23.5, -46.0, 113, 128),
 (-23.5, -45.5, 113, 129),
 (-23.0, -46.0, 114, 128),
 (-23.0, -45.5, 114, 129)]

In [ ]:
maggia_tec.tec_map = np_mapas_maggia[0]

print("Os 4 pontos mais próximos são:")
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-23.5°, -46.0°) - Índices: [113, 128] - Valor TEC: 81.59700012207031
Coordenadas: (-23.5°, -45.5°) - Índices: [113, 129] - Valor TEC: 82.16899871826172
Coordenadas: (-23.0°, -46.0°) - Índices: [114, 128] - Valor TEC: 86.26799774169922
Coordenadas: (-23.0°, -45.5°) - Índices: [114, 129] - Valor TEC: 85.58100128173828

Valor TEC interpolado para (-23.21°, -45.51°): 84.1511


## 12/18/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-18'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_maggia_2024[df_mapas_maggia_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_maggia_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_maggia_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_maggia_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
13716,2024-12-18 18:00:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


In [ ]:
mapas_maggia = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_maggia = []
for i in range(len(mapas_maggia)):
    np_mapas_maggia.append(mapas_maggia[i])
np_mapas_maggia = np.array(np_mapas_maggia)

In [ ]:
np_mapas_maggia

array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [34.4980011 , 34.72600174, 34.9109993 , ..., 13.43500042,
         13.27900028, 12.76099968],
        [34.2859993 , 34.53300095, 34.74300003, ..., 13.2489996 ,
         12.7510004 , 12.31400013],
        [34.10699844, 34.35800171, 34.58399963, ..., 12.86299992,
         12.45899963, 12.0880003 ]]])

In [ ]:
np_mapas_maggia.shape

(1, 241, 221)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


maggia_tec = maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(maggia_points)

[(-23.5, -46.0, 113, 128), (-23.5, -45.5, 113, 129), (-23.0, -46.0, 114, 128), (-23.0, -45.5, 114, 129)]


In [ ]:
maggia_points

[(-23.5, -46.0, 113, 128),
 (-23.5, -45.5, 113, 129),
 (-23.0, -46.0, 114, 128),
 (-23.0, -45.5, 114, 129)]

In [ ]:
maggia_tec.tec_map = np_mapas_maggia[0]

print("Os 4 pontos mais próximos são:")
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-23.5°, -46.0°) - Índices: [113, 128] - Valor TEC: 85.70800018310547
Coordenadas: (-23.5°, -45.5°) - Índices: [113, 129] - Valor TEC: 79.35399627685547
Coordenadas: (-23.0°, -46.0°) - Índices: [114, 128] - Valor TEC: 82.80599975585938
Coordenadas: (-23.0°, -45.5°) - Índices: [114, 129] - Valor TEC: 83.21299743652344

Valor TEC interpolado para (-23.21°, -45.51°): 81.6409


## 12/22/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-22'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_maggia_2024[df_mapas_maggia_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_maggia_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_maggia_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_maggia_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
14292,2024-12-22 18:00:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


In [ ]:
mapas_maggia = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_maggia = []
for i in range(len(mapas_maggia)):
    np_mapas_maggia.append(mapas_maggia[i])
np_mapas_maggia = np.array(np_mapas_maggia)

In [ ]:
np_mapas_maggia

array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [48.41400146, 48.5379982 , 48.59199905, ..., 19.10899925,
         17.45599937, 17.45599937],
        [48.26200104, 48.41799927, 48.52799988, ..., 18.8560009 ,
         18.24600029, 17.79299927],
        [48.11100006, 48.27999878, 48.42100143, ..., 19.0890007 ,
         18.74200058, 18.39999962]]])

In [ ]:
np_mapas_maggia.shape

(1, 241, 221)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


maggia_tec = maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(maggia_points)

[(-23.5, -46.0, 113, 128), (-23.5, -45.5, 113, 129), (-23.0, -46.0, 114, 128), (-23.0, -45.5, 114, 129)]


In [ ]:
maggia_points

[(-23.5, -46.0, 113, 128),
 (-23.5, -45.5, 113, 129),
 (-23.0, -46.0, 114, 128),
 (-23.0, -45.5, 114, 129)]

In [ ]:
maggia_tec.tec_map = np_mapas_maggia[0]

print("Os 4 pontos mais próximos são:")
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-23.5°, -46.0°) - Índices: [113, 128] - Valor TEC: 87.62000274658203
Coordenadas: (-23.5°, -45.5°) - Índices: [113, 129] - Valor TEC: 86.12300109863281
Coordenadas: (-23.0°, -46.0°) - Índices: [114, 128] - Valor TEC: 87.91799926757812
Coordenadas: (-23.0°, -45.5°) - Índices: [114, 129] - Valor TEC: 86.75599670410156

Valor TEC interpolado para (-23.21°, -45.51°): 86.5162


## 12/19/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-19'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_maggia_2024[df_mapas_maggia_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_maggia_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_maggia_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_maggia_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
13860,2024-12-19 18:00:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


In [ ]:
mapas_maggia = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_maggia = []
for i in range(len(mapas_maggia)):
    np_mapas_maggia.append(mapas_maggia[i])
np_mapas_maggia = np.array(np_mapas_maggia)

In [ ]:
np_mapas_maggia

array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [37.94499969, 38.06100082, 38.13499832, ..., 11.56599998,
         11.17399979, 10.60200024],
        [37.7840004 , 37.92599869, 38.02999878, ..., 11.47599983,
         10.99199963, 10.38300037],
        [37.63499832, 37.77899933, 37.90499878, ..., 11.36100006,
         10.8739996 , 10.52200031]]])

In [ ]:
np_mapas_maggia.shape

(1, 241, 221)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


maggia_tec = maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(maggia_points)

[(-23.5, -46.0, 113, 128), (-23.5, -45.5, 113, 129), (-23.0, -46.0, 114, 128), (-23.0, -45.5, 114, 129)]


In [ ]:
maggia_points

[(-23.5, -46.0, 113, 128),
 (-23.5, -45.5, 113, 129),
 (-23.0, -46.0, 114, 128),
 (-23.0, -45.5, 114, 129)]

In [ ]:
maggia_tec.tec_map = np_mapas_maggia[0]

print("Os 4 pontos mais próximos são:")
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-23.5°, -46.0°) - Índices: [113, 128] - Valor TEC: 80.61499786376953
Coordenadas: (-23.5°, -45.5°) - Índices: [113, 129] - Valor TEC: 80.79199981689453
Coordenadas: (-23.0°, -46.0°) - Índices: [114, 128] - Valor TEC: 80.12999725341797
Coordenadas: (-23.0°, -45.5°) - Índices: [114, 129] - Valor TEC: 80.7760009765625

Valor TEC interpolado para (-23.21°, -45.51°): 80.7737


## 12/01/2024 - 18:00

In [ ]:
import pandas as pd

data = '2024-12-01'
horario = '18:00:00'
timestamp_procurado = f'{data} {horario}'

resultado = df_mapas_maggia_2024[df_mapas_maggia_2024['DATETIME'] == timestamp_procurado]

if resultado.empty:
    df_temp = df_mapas_maggia_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_procurado)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_mais_proximo = df_temp['diff'].idxmin()

    resultado = df_mapas_maggia_2024.iloc[[idx_mais_proximo]]
    print(f"Data exata não encontrada. Usando o registro mais próximo: {df_mapas_maggia_2024.iloc[idx_mais_proximo]['DATETIME']}")

In [ ]:
resultado

,DATETIME,TECMAP
11268,2024-12-01 18:00:00,"[[nan, nan, nan, nan, nan, nan, nan, nan, nan,..."


In [ ]:
mapas_maggia = np.array(resultado.iloc[:]['TECMAP'])

In [ ]:
np_mapas_maggia = []
for i in range(len(mapas_maggia)):
    np_mapas_maggia.append(mapas_maggia[i])
np_mapas_maggia = np.array(np_mapas_maggia)

In [ ]:
np_mapas_maggia

array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [45.95299911, 45.88100052, 45.71300125, ..., 23.45899963,
         23.20400047, 22.93400002],
        [45.90299988, 45.8370018 , 45.67300034, ..., 23.38800049,
         23.23900032, 23.04000092],
        [45.86399841, 45.79700089, 45.63999939, ..., 23.29700089,
         23.1590004 , 23.00300026]]])

In [ ]:
np_mapas_maggia.shape

(1, 241, 221)

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


maggia_tec = maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(maggia_points)

[(-23.5, -46.0, 113, 128), (-23.5, -45.5, 113, 129), (-23.0, -46.0, 114, 128), (-23.0, -45.5, 114, 129)]


In [ ]:
maggia_points

[(-23.5, -46.0, 113, 128),
 (-23.5, -45.5, 113, 129),
 (-23.0, -46.0, 114, 128),
 (-23.0, -45.5, 114, 129)]

In [ ]:
maggia_tec.tec_map = np_mapas_maggia[0]

print("Os 4 pontos mais próximos são:")
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordenadas: ({lat}°, {lon}°) - Índices: [{lat_idx}, {lon_idx}] - Valor TEC: {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nValor TEC interpolado para ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

Os 4 pontos mais próximos são:
Coordenadas: (-23.5°, -46.0°) - Índices: [113, 128] - Valor TEC: 86.64099884033203
Coordenadas: (-23.5°, -45.5°) - Índices: [113, 129] - Valor TEC: 85.81099700927734
Coordenadas: (-23.0°, -46.0°) - Índices: [114, 128] - Valor TEC: 89.6050033569336
Coordenadas: (-23.0°, -45.5°) - Índices: [114, 129] - Valor TEC: 88.98799896240234

Valor TEC interpolado para (-23.21°, -45.51°): 87.6678
